In [1]:
# imports
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

print("Pandas version:", pd.__version__)
print("Numpy version:", np.__version__)

Pandas version: 3.0.5
Numpy version: 2.5.1


## 1. Load Processed Dataset

To initiate the modeling workflow, we import the fully cleaned, encoded, and engineered dataset from our central `data/processed/` directory.

* **Pipeline Verification:** Inspecting the first few records and schema metadata ensures that feature dtypes, one-hot encoded columns, and engineered variables (`Total_Debt_Exposure`) were preserved accurately during export.
* **Integrity Check:** Confirms row counts, feature counts, and non-null values prior to feature scaling or model initialization.

In [ ]:
# Load Processed dataset
X_train = pd.read_csv("../data/processed/X_train_processed.csv")
X_test = pd.read_csv("../data/processed/X_test_processed.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze("columns")
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze("columns")
# Inspect all datasets
print("\nX_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)


### 🔍 Dataset Loading & Schema Verification

The processed dataset was imported successfully and verified against expected pipeline specifications:

* **Dimensions:** $20,000$ rows across $44$ features.
* **Feature Consistency:** Includes original numerical features, engineered metrics (`Total_Debt_Exposure`), and one-hot encoded categorical indicators.
* **Data Readiness:** All data types are aligned for numerical operations, with zero remaining null or unencoded categorical values.

> **Status:** The dataset structure is validated and ready for model partitioning and baseline training.

## Cross-Validation Evaluation



Before training our final baseline model on the complete training set, we perform **5-fold cross-validation** to assess model generalization and stability across different subsets of $X_{\text{train}}$.

* **5-Fold Split ($k=5$):** $X_{\text{train}}$ ($16,000$ samples) is partitioned into $5$ equal folds ($3,200$ samples per fold). The model trains on $4$ folds ($12,800$ samples) and validates on the remaining fold in $5$ sequential iterations.
* **Metric Choice (`scoring="accuracy"`):** Measures the proportion of correctly classified loan statuses in each validation fold.
* **Data Leakage Isolation:** Cross-validation is executed strictly on $X_{\text{train}}$ and $y_{\text{train}}$. The holdout test set ($X_{\text{test}}$, $y_{\text{test}}$) remains entirely untouched.

In [ ]:
# ============================================================
#  CROSS-VALIDATION
# ============================================================




# ------------------------------------------------------------
#  Create the Decision Tree model
# ------------------------------------------------------------
# We create the model here so that cross-validation can
# train and evaluate it across different folds of X_train.
#
# random_state=42 ensures that the results are reproducible.
# ------------------------------------------------------------

model = DecisionTreeClassifier(random_state=42)


# ------------------------------------------------------------
# Perform 5-Fold Cross-Validation
# ------------------------------------------------------------
# cv=5 means that the training data will be divided into
# 5 folds.
#
# The model will be trained 5 different times.
# In each round, 4 folds are used for training and 1 fold
# is used for validation.
#
# scoring="accuracy" tells sklearn to measure the percentage
# of correct predictions in each validation fold.
#
# IMPORTANT:
# We use X_train and y_train only.
# The test data (X_test and y_test) is NOT used here.
# ------------------------------------------------------------

cv_scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy"
)


# ------------------------------------------------------------
#  Display the accuracy from each fold
# ------------------------------------------------------------

print("=" * 60)
print("CROSS-VALIDATION RESULTS")
print("=" * 60)

print("\nAccuracy for each fold:")

for i, score in enumerate(cv_scores, start=1):
    print(f"Fold {i}: {score:.4f}")


# ------------------------------------------------------------
#  Calculate the mean cross-validation accuracy
# ------------------------------------------------------------
# The mean gives us the average performance of the model
# across all 5 validation folds.
# ------------------------------------------------------------

mean_cv_score = cv_scores.mean()

print("\nMean Cross-Validation Accuracy:")
print(f"{mean_cv_score:.4f}")


# ------------------------------------------------------------
#  Calculate the standard deviation
# ------------------------------------------------------------
# Standard deviation tells us how much the model's
# performance varies between the different folds.
#
# A smaller standard deviation generally means that the
# model's performance is more consistent across the folds.
# ------------------------------------------------------------

std_cv_score = cv_scores.std()

print("\nStandard Deviation of CV Accuracy:")
print(f"{std_cv_score:.4f}")


# ------------------------------------------------------------
# Display the final summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CROSS-VALIDATION SUMMARY")
print("=" * 60)

print(f"Number of folds: 5")
print(f"Individual fold accuracies: {cv_scores}")
print(f"Mean CV accuracy: {mean_cv_score:.4f}")
print(f"Standard deviation: {std_cv_score:.4f}")

print("\nCross-validation completed successfully.")

### 📊 Cross-Validation Results & Fold Breakdown

The 5-fold cross-validation results show consistent performance across all validation subsets:

#### Fold Accuracy Breakdown
* **Fold 1:** $0.9416$ ($94.16\%$)
* **Fold 2:** $0.9319$ ($93.19\%$)
* **Fold 3:** $0.9406$ ($94.06\%$)
* **Fold 4:** $0.9334$ ($93.34\%$)
* **Fold 5:** $0.9444$ ($94.44\%$)

#### Overall Performance Summary

| Metric | Value | Interpretation |
| :--- | :--- | :--- |
| **Mean CV Accuracy** | **$0.9384$ ($93.84\%$)** | High average baseline performance across all holdout folds |
| **Standard Deviation ($\sigma$)** | **$0.0049$ ($0.49\%$)** | Exceptionally low variance, confirming model stability |

> **Key Takeaway:** The tight clustering of scores between $93.19\%$ and $94.44\%$ yields a small standard deviation ($0.0049$), proving the Decision Tree performs reliably regardless of data sub-sampling. We can now safely fit our final model on the full $X_{\text{train}}$ dataset.

**Notes:**

Cross-validation helps us determine whether the Decision Tree is performing reliably or just happened to perform well on one particular split of the training data.

Cross-validation is used to check how consistently the model performs across different portions of the training data.

Instead of relying on one train/validation split, it tests the model multiple times and gives us an average performance score.

##  Final Model Training


Having validated baseline model stability through 5-fold cross-validation, we fit the `DecisionTreeClassifier` on the complete training set ($X_{\text{train}}$, $y_{\text{train}}$).

* **Maximizing Learning:** Fitting across $100\%$ of the training data ($16,000$ samples) allows the tree to build split rules using the maximum amount of information available.
* **Preserving Integrity:** The test set ($X_{\text{test}}$, $y_{\text{test}}$) remains strictly segregated for out-of-sample evaluation.

In [ ]:


# Train the Decision Tree model using the entire training dataset.
# X_train contains the input features.
# y_train contains the target variable (Loan_Status).

model.fit(X_train, y_train)


# ------------------------------------------------------------
# Display training completion message
# ------------------------------------------------------------

print("=" * 60)
print("MODEL TRAINING")
print("=" * 60)

print("\nDecision Tree model trained successfully.")

print(f"Number of training samples: {X_train.shape[0]}")
print(f"Number of features: {X_train.shape[1]}")

### 🎯 Model Training Summary

The `DecisionTreeClassifier` was successfully fitted on the full training partition:

* **Training Samples:** $16,000$ rows ($80.00\%$ of total dataset)
* **Feature Space:** $43$ predictor variables
* **Model Status:** Fitted and ready for predictions on unseen holdout data (`X_test`)

> **Next Step:** Generate predictions on $X_{\text{test}}$ to evaluate out-of-sample generalization metrics including accuracy, precision, recall, F1-score, and the confusion matrix.

## Final Model Evaluation



To quantify out-of-sample generalization, we generate predictions using the holdout test set ($X_{\text{test}}$, $y_{\text{test}}$), which was strictly isolated from training and cross-validation.

* **Generalization Check:** Compares performance on unseen data against the $5$-fold cross-validation mean ($93.84\%$) to check for overfitting.
* **Evaluation Framework:** Uses **Accuracy**, **Confusion Matrix**, and **Classification Report** (Precision, Recall, F1-Score) to assess class-level predictive quality.

In [ ]:
# ============================================================
# FINAL EVALUATION
# ============================================================



# ------------------------------------------------------------
# Make predictions on the unseen test data
# ------------------------------------------------------------
# The model has already been trained using X_train and y_train.
# Now we give it X_test, which it has never seen before.
# ------------------------------------------------------------

y_pred = model.predict(X_test)


# ------------------------------------------------------------
# Calculate the test accuracy
# ------------------------------------------------------------
# Accuracy tells us the percentage of test predictions
# that the model got correct.
# ------------------------------------------------------------

test_accuracy = accuracy_score(y_test, y_pred)


# ------------------------------------------------------------
# Display the final accuracy
# ------------------------------------------------------------

print("=" * 60)
print("FINAL MODEL EVALUATION")
print("=" * 60)

print(f"\nTest Accuracy: {test_accuracy:.4f}")
print(f"Test Accuracy (%): {test_accuracy * 100:.2f}%")


# ------------------------------------------------------------
# Display the confusion matrix
# ------------------------------------------------------------
# The confusion matrix shows:
#
# True Negatives  → Correctly predicted negative class
# False Positives → Negative class predicted as positive
# False Negatives → Positive class predicted as negative
# True Positives  → Correctly predicted positive class
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)

cm = confusion_matrix(y_test, y_pred)

print(cm)


# ------------------------------------------------------------
# Display the classification report
# ------------------------------------------------------------
# The classification report provides:
#
# Precision → How many predicted positives were actually positive
# Recall    → How many actual positives were correctly identified
# F1-score  → Balance between precision and recall
# Support   → Number of actual samples in each class
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)

print(classification_report(y_test, y_pred))

### 📊 Test Evaluation Results & Interpretation

The Decision Tree model achieved **$93.70\%$ accuracy** on unseen holdout test data, demonstrating exceptional alignment with the 5-fold cross-validation mean score ($93.84\%$). This confirms that the model generalizes well without overfitting.

---

#### 📌 Confusion Matrix Breakdown ($N = 4,000$)

| | Predicted: Class 0 (Rejected) | Predicted: Class 1 (Approved) | Total Actual |
| :--- | :---: | :---: | :---: |
| **Actual: Class 0 (Rejected)** | **1,299** (True Negatives) | **130** (False Positives) | **1,429** |
| **Actual: Class 1 (Approved)** | **122** (False Negatives) | **2,449** (True Positives) | **2,571** |

* **True Negatives ($TN = 1,299$):** Correctly predicted rejected applications.
* **True Positives ($TP = 2,449$):** Correctly predicted approved applications.
* **False Positives ($FP = 130$):** Rejected loans incorrectly classified as approved (Type I Error).
* **False Negatives ($FN = 122$):** Approved loans incorrectly classified as rejected (Type II Error).

---

#### 🎯 Classification Performance Metrics

* **Overall Accuracy ($93.70\%$):** $3,748$ out of $4,000$ test cases were correctly classified.
* **Class 0 (Rejected Loans):**
  * **Precision ($0.91$):** When the model predicts a rejection, it is correct $91\%$ of the time.
  * **Recall ($0.91$):** The model successfully identifies $91\%$ of all actual rejections.
  * **F1-Score ($0.91$):** Strong balance between precision and recall for the minority class.
* **Class 1 (Approved Loans):**
  * **Precision ($0.95$):** When the model predicts an approval, it is correct $95\%$ of the time.
  * **Recall ($0.95$):** The model captures $95\%$ of all actual loan approvals.
  * **F1-Score ($0.95$):** Highly reliable performance on the majority class.

---

#### 💡 Key Takeaways

1. **No Overfitting:** The test accuracy ($93.70\%$) matches the cross-validation score ($93.84\%$) within a tiny margin ($0.14\%$), indicating robust generalization.
2. **Balanced Errors:** False positives ($130$) and false negatives ($122$) occur at nearly equal rates, indicating no systematic class bias.
3. **High Operational Utility:** With $91\%+$ precision and recall across both classes, the model is well-suited for automated decisioning or triage pipelines.

## Save Trained Model



To persist our trained `DecisionTreeClassifier` for downstream deployment, API integration, or batch inference, we serialize the model object to disk using `joblib`.

* **Path Traversal (`../models/`):** Navigates up from `notebooks/` into the central `models/` folder to separate model artifacts from code and data assets.
* **Serialization Choice (`joblib`):** Efficiently serializes large Python data structures and scikit-learn models containing heavy NumPy arrays.

In [ ]:
# ============================================================
# SAVE TRAINED MODEL
# ============================================================




# Save the trained Decision Tree model
joblib.dump(model, "../models/loan_approval_decision_tree.pkl")


# Confirm that the model was saved successfully
print("=" * 60)
print("MODEL SAVING")
print("=" * 60)

print("\nModel saved successfully.")
print("File: loan_approval_decision_tree.pkl")

### 💾 Model Serialization Summary

The trained Decision Tree model artifact has been written to disk:

* **Saved Artifact:** `models/loan_approval_decision_tree.pkl`
* **Model Configuration:** `DecisionTreeClassifier(random_state=42)`
* **Deployment Readiness:** The `.pkl` binary can now be loaded directly into inference pipelines using `joblib.load()` without retraining.

> **Project Milestone Reached:** The end-to-end Machine Learning pipeline—spanning data cleaning, feature engineering, train-test splitting, cross-validation, model training, test evaluation, and artifact serialization—is complete!